# 🏦 BFSI AI Multi-Agent Advisor
### Banking · Finance · Securities · Insurance

**Open-Source LLM:** Groq LLaMA 3.1 8B (fast) + LLaMA 3.3 70B (reports)

---

### 🤖 6 Specialist AI Agents
| # | Agent | Covers |
|---|-------|--------|
| 1 | 🔍 Profile Analyst | Risk scoring, financial health, strengths & concerns |
| 2 | 💳 Credit Advisor | Loan eligibility, EMI, CIBIL tips, product picks |
| 3 | 📈 Investment Strategist | Asset allocation, mutual funds, SIP, expected returns |
| 4 | 🛡️ Insurance Consultant | Term, health, critical illness, premiums, 80D |
| 5 | 🧾 Tax Expert | Old vs New regime, 80C/80D, ITR guidance |
| 6 | 📚 Literacy Coach | Concepts, fraud alerts, 30-60-90 day action plan |

---
### ⚙️ Setup: 3 Steps
1. **Cell 1** → Install packages
2. **Cell 2** → Paste your Groq API key (free at https://console.groq.com)
3. **Cell 3** → Launch the app


In [1]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 1 — Install Dependencies              ║
# ╚══════════════════════════════════════════════╝
!pip install -q groq gradio requests
print("✅ All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.7 MB/s eta 0:00:00
✅ All packages installed successfully!


In [2]:
# ╔══════════════════════════════════════════════╗
# ║  CELL 2 — Paste Your API Keys Here          ║
# ╚══════════════════════════════════════════════╝

GROQ_API_KEY   = "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"      # 🔑 Required — get free at console.groq.com
SERPER_API_KEY = "34cf9bb4a6965f4ad9bcce86f48f5d693048ac09"         # 🔎 Optional — enables live market data

print("🔑 API Keys configured!")
print(f"   Groq   : {'✅ Set' if GROQ_API_KEY.startswith('gsk_') and len(GROQ_API_KEY) > 20 else '❌ Not set — please update GROQ_API_KEY'}")
print(f"   Serper : {'✅ Set' if len(SERPER_API_KEY) > 10 and SERPER_API_KEY != 'YOUR_SERPER_KEY_HERE' else '⚠️  Not set — live search disabled'}")

🔑 API Keys configured!
   Groq   : ✅ Set
   Serper : ✅ Set


In [3]:
import os, sys, json, re
from dataclasses import dataclass
from typing import List, Dict, Tuple
import requests
import gradio as gr
from groq import Groq

In [4]:
# ── Configuration ──────────────────────────────────────────────────────
GROQ_MODEL    = "llama-3.1-8b-instant"       # fast open-source model
GROQ_MODEL_HQ = "llama-3.3-70b-versatile"   # high-quality for reports
groq_client   = Groq(api_key=GROQ_API_KEY)

In [5]:
# ── STEP 1: Groq LLM Helper ────────────────────────────────────────────
def call_groq(system: str, user: str, max_tokens: int = 800, use_hq: bool = False) -> str:
    """Call Groq open-source LLM. Never raises — returns error string on failure."""
    model = GROQ_MODEL_HQ if use_hq else GROQ_MODEL
    try:
        resp = groq_client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
            max_tokens=max_tokens,
            temperature=0.2,
        )
        return resp.choices[0].message.content.strip()
    except Exception as exc:
        return f"⚠️ LLM Error: {exc}"

In [6]:
# ── STEP 2: Live Market Data (Serper, optional) ────────────────────────
def fetch_market_data(query: str) -> str:
    if not SERPER_API_KEY or SERPER_API_KEY == "YOUR_SERPER_KEY_HERE":
        return "(Live market search not available — add SERPER_API_KEY to enable)"
    try:
        resp = requests.post(
            "https://google.serper.dev/search",
            json={"q": query + " India 2025", "gl": "IN", "hl": "en", "num": 4},
            headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
            timeout=8,
        )
        resp.raise_for_status()
        items = resp.json().get("organic", [])[:3]
        return "\n".join(f"• {r['title']}: {r.get('snippet','')}" for r in items)
    except Exception as exc:
        return f"(Search error: {exc})"

In [7]:


# ── STEP 3: BFSI Agents ────────────────────────────────────────────────

@dataclass
class ProfileAnalystAgent:
    """Agent 1: Parses financial profile and assesses risk appetite."""
    role: str = "BFSI Profile Analyst"
    goal: str = "Parse financial profile and assess risk appetite accurately"

    def run(self, age, income, savings, debt, goal, horizon, risk) -> Dict:
        system = (
            "You are a senior BFSI analyst. Analyse the client profile and return a "
            "structured JSON with keys: risk_category (Conservative/Moderate/Aggressive), "
            "financial_health (Poor/Fair/Good/Excellent), key_strengths (list), "
            "key_concerns (list), priority_action (string). Return ONLY valid JSON."
        )
        user = (
            f"Age: {age} | Monthly Income: ₹{income} | Savings: ₹{savings}\n"
            f"Existing EMI: ₹{debt}/month | Goal: {goal}\n"
            f"Investment Horizon: {horizon} | Self-assessed Risk: {risk}"
        )
        raw = call_groq(system, user, max_tokens=350)
        try:
            clean = re.sub(r"```[\w]*\n?", "", raw).strip()
            return json.loads(clean)
        except Exception:
            return {
                "risk_category": risk, "financial_health": "Fair",
                "key_strengths": ["Regular income"], "key_concerns": ["Needs full analysis"],
                "priority_action": "Consult a certified financial planner."
            }

In [8]:

@dataclass
class CreditAdvisorAgent:
    """Agent 2: Evaluates loan eligibility, EMI, and credit improvement tips."""
    role: str = "Credit & Loan Specialist"
    goal: str = "Assess loan eligibility and recommend best credit products"

    def run(self, income, debt, savings, goal, profile) -> str:
        market = fetch_market_data("best home loan personal loan interest rates India banks 2026")
        system = (
            "You are a credit specialist at a leading Indian bank. Provide structured credit "
            "advice: loan eligibility, recommended products with interest rates, EMI estimates, "
            "CIBIL score improvement tips, and red flags. Use ₹. Be specific with numbers."
        )
        user = (
            f"Profile: {json.dumps(profile)}\n"
            f"Monthly Income: ₹{income} | Existing EMI: ₹{debt} | Savings: ₹{savings}\n"
            f"Goal: {goal}\nMarket Data:\n{market}\n\n"
            "Provide detailed credit advisory with actionable steps and specific bank names."
        )
        return call_groq(system, user, max_tokens=900, use_hq=True)

In [9]:

@dataclass
class InvestmentStrategistAgent:
    """Agent 3: Portfolio allocation across equity, debt, gold, real estate."""
    role: str = "Investment Strategist"
    goal: str = "Build optimal investment portfolio for risk and time horizon"

    def run(self, income, savings, horizon, goal, profile) -> str:
        market = fetch_market_data("best mutual funds SIP equity debt returns India 2026")
        system = (
            "You are a SEBI-registered investment advisor. Provide: "
            "1) Asset allocation % (equity/debt/gold/liquid), "
            "2) Specific Indian fund recommendations with fund names, "
            "3) SIP amounts, 4) Expected returns over horizon, "
            "5) Tax implications. Add a brief disclaimer at end."
        )
        user = (
            f"Risk: {profile.get('risk_category','Moderate')} | Health: {profile.get('financial_health','Fair')}\n"
            f"Income: ₹{income}/month | Savings: ₹{savings} | Goal: {goal} | Horizon: {horizon}\n"
            f"Market Context:\n{market}\n\n"
            "Design a complete investment portfolio with specific fund names and allocation percentages."
        )
        return call_groq(system, user, max_tokens=1000, use_hq=True)

In [10]:

@dataclass
class InsuranceConsultantAgent:
    """Agent 4: Life, health, term, and critical illness recommendations."""
    role: str = "Insurance Consultant"
    goal: str = "Recommend complete insurance coverage for total protection"

    def run(self, age, income, goal, dependents, profile) -> str:
        market = fetch_market_data("best term insurance health insurance plans premiums India 2026")
        system = (
            "You are an IRDAI-licensed insurance advisor. Recommend: "
            "1) Term life insurance (cover amount + top 3 plans with premiums), "
            "2) Health/family floater insurance (top plans), "
            "3) Critical illness cover, 4) Accidental cover, "
            "5) Premium estimates, 6) Tax benefits under 80C/80D. "
            "Name real Indian insurers and products."
        )
        user = (
            f"Age: {age} | Income: ₹{income}/month | Dependents: {dependents}\n"
            f"Goal: {goal} | Risk: {profile.get('risk_category','Moderate')}\n"
            f"Market Context:\n{market}\n\n"
            "Provide complete insurance needs analysis with specific product names and premium estimates."
        )
        return call_groq(system, user, max_tokens=900, use_hq=True)

In [11]:
@dataclass
class TaxComplianceAgent:
    """Agent 5: Tax regime comparison, 80C/80D deductions, ITR guidance."""
    role: str = "Tax & Compliance Expert"
    goal: str = "Maximise tax savings through legal instruments and planning"

    def run(self, income, savings, age, goal, profile) -> str:
        try:
            annual = int(str(income).replace(",","")) * 12
            annual_fmt = f"₹{annual:,}"
        except Exception:
            annual_fmt = f"₹{income} x 12"
        system = (
            "You are a Chartered Accountant specialising in Indian personal taxation. "
            "Provide: 1) Old vs New Tax Regime comparison with exact tax amounts, "
            "2) 80C instruments (ELSS/PPF/NPS/ULIP) with optimal amounts, "
            "3) 80D health insurance deduction, "
            "4) HRA, LTA, standard deduction benefits, "
            "5) Estimated total tax savings, 6) ITR filing tips. "
            "Use FY 2024-25 tax slabs. Be precise with ₹ amounts."
        )
        user = (
            f"Annual Income: {annual_fmt} | Age: {age}\n"
            f"Goal: {goal} | Annual Savings Capacity: ₹{savings}\n"
            f"Profile: {json.dumps(profile)}\n\n"
            "Provide complete tax optimisation plan for FY 2024-25 with exact numbers."
        )
        return call_groq(system, user, max_tokens=1000, use_hq=True)

In [12]:

@dataclass
class FinancialLiteracyCoachAgent:
    """Agent 6: Financial literacy, fraud alerts, and 30-60-90 day action plan."""
    role: str = "Financial Literacy Coach"
    goal: str = "Educate, alert, and empower with financial knowledge"

    def run(self, goal, horizon, profile) -> str:
        system = (
            "You are a financial wellness coach and fraud prevention expert for Indians. "
            "Provide: 1) 5 key financial concepts this client should master, "
            "2) Top 5 BFSI frauds to watch for (UPI fraud, KYC scam, loan fraud, etc.), "
            "3) A 30-60-90 day personalised financial action plan with specific tasks, "
            "4) Top 5 free financial tools/apps for Indians (names + what they do), "
            "5) An empowering closing message. Keep it practical and actionable."
        )
        user = (
            f"Goal: {goal} | Horizon: {horizon}\n"
            f"Health: {profile.get('financial_health','Fair')} | Risk: {profile.get('risk_category','Moderate')}\n"
            f"Priority Action: {profile.get('priority_action','')}\n\n"
            "Create a personalised financial literacy guide and action plan."
        )
        return call_groq(system, user, max_tokens=900, use_hq=True)

In [13]:

# ── STEP 4: Orchestrator ───────────────────────────────────────────────
def bfsi_advisor(age, income, savings, debt, dependents, goal, horizon, risk):
    """Run all 6 agents. Returns 6 Markdown strings for 6 output tabs."""
    if not all(s.strip() for s in [age, income, savings, goal, horizon, risk]):
        msg = "⚠️ Please fill in all required fields (Age, Income, Savings, Goal, Horizon, Risk Appetite)."
        return msg, msg, msg, msg, msg, msg

    income   = str(income).replace(",","").replace("₹","").strip()
    debt     = str(debt).strip() or "0"
    deps     = str(dependents).strip() or "None"

    # Agent 1 — Profile
    analyst = ProfileAnalystAgent()
    profile = analyst.run(age, income, savings, debt, goal, horizon, risk)

    he = {"Poor":"🔴","Fair":"🟡","Good":"🟢","Excellent":"✨"}.get(profile.get("financial_health","Fair"),"🟡")
    re_ = {"Conservative":"🛡️","Moderate":"⚖️","Aggressive":"🚀"}.get(profile.get("risk_category",risk),"⚖️")

    profile_md = f"""## 🔍 Your Financial Profile Analysis

| Attribute | Assessment |
|-----------|------------|
| **Financial Health** | {he} {profile.get("financial_health","Fair")} |
| **Risk Category** | {re_} {profile.get("risk_category",risk)} |
| **Investment Horizon** | 🗓️ {horizon} |
| **Primary Goal** | 🎯 {goal} |

### ✅ Key Strengths
{chr(10).join(f"- {s}" for s in profile.get("key_strengths",[]))}

### ⚠️ Areas to Address
{chr(10).join(f"- {c}" for c in profile.get("key_concerns",[]))}

### 🎯 Priority Action
> {profile.get("priority_action","Consult a certified financial planner.")}

---
*AI-generated profile — consult a SEBI/IRDAI registered advisor for actual decisions.*"""

    # Agent 2 — Credit
    credit_md = "## 💳 Credit & Loan Advisory\n\n" + CreditAdvisorAgent().run(income, debt, savings, goal, profile)

    # Agent 3 — Investment
    invest_md = "## 📈 Investment Portfolio Strategy\n\n" + InvestmentStrategistAgent().run(income, savings, horizon, goal, profile)

    # Agent 4 — Insurance
    insure_md = "## 🛡️ Insurance Coverage Recommendations\n\n" + InsuranceConsultantAgent().run(age, income, goal, deps, profile)

    # Agent 5 — Tax
    tax_md = "## 🧾 Tax Optimisation Plan\n\n" + TaxComplianceAgent().run(income, savings, age, goal, profile)

    # Agent 6 — Coach
    coach_md = "## 📚 Financial Literacy & Action Plan\n\n" + FinancialLiteracyCoachAgent().run(goal, horizon, profile)

    return profile_md, credit_md, invest_md, insure_md, tax_md, coach_md

In [14]:

# ── STEP 5: CSS — Luxury Dark Gold Aesthetic ──────────────────────────
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Cormorant+Garamond:ital,wght@0,400;0,600;0,700;1,400&family=Outfit:wght@300;400;500;600&display=swap');
:root {
  --gold:#C9A84C; --gold-light:#E8C96A; --gold-dim:#8A6E30;
  --bg-deep:#080C14; --bg-card:#0E1520; --bg-input:#F7F3EC;
  --text-main:#EDE5D4; --text-muted:#8A8070;
  --border:rgba(201,168,76,0.2);
}
.gradio-container { background:var(--bg-deep) !important; font-family:'Outfit',sans-serif !important; min-height:100vh; }
.gold-rule { height:3px; background:linear-gradient(90deg,transparent,var(--gold),transparent); margin:0 0 18px; animation:shimmer 3s ease-in-out infinite; }
@keyframes shimmer { 0%,100%{opacity:.6} 50%{opacity:1} }
#bfsi-title { font-family:'Cormorant Garamond',serif !important; font-size:2.8rem !important; font-weight:700 !important; letter-spacing:2px; color:var(--gold-light) !important; text-shadow:0 0 40px rgba(201,168,76,.4); margin:0 !important; text-align:center; }
#bfsi-subtitle { font-size:.85rem; font-weight:300; color:var(--text-muted); letter-spacing:3px; text-transform:uppercase; text-align:center; margin:5px 0 16px; }
#bfsi-subtitle span { color:var(--gold); }
.agents-strip { display:flex; gap:8px; justify-content:center; flex-wrap:wrap; padding:12px; background:rgba(201,168,76,.04); border-top:1px solid var(--border); border-bottom:1px solid var(--border); margin-bottom:20px; }
.agent-badge { display:inline-flex; align-items:center; gap:5px; background:rgba(201,168,76,.08); border:1px solid rgba(201,168,76,.25); border-radius:20px; padding:5px 13px; font-size:.76rem; color:var(--gold); font-weight:500; }
.section-label { font-size:.7rem; font-weight:600; color:var(--gold-dim); text-transform:uppercase; letter-spacing:2px; margin:14px 0 7px; padding-left:2px; }
.gradio-container input,.gradio-container textarea,.gradio-container select,input,textarea,select { background-color:var(--bg-input) !important; color:#1A1208 !important; border:1px solid rgba(201,168,76,.3) !important; border-radius:8px !important; font-family:'Outfit',sans-serif !important; font-size:.9rem !important; padding:9px 13px !important; caret-color:var(--gold) !important; }
.gradio-container input:focus,.gradio-container textarea:focus,input:focus,textarea:focus { border-color:var(--gold) !important; box-shadow:0 0 0 2px rgba(201,168,76,.15) !important; outline:none !important; color:#1A1208 !important; background-color:#FFFDF5 !important; }
input::placeholder,textarea::placeholder { color:#8B7A5A !important; opacity:1 !important; }
.gradio-container label>span,.gradio-container .label-wrap>span { font-family:'Outfit',sans-serif !important; font-size:.73rem !important; font-weight:600 !important; color:var(--gold) !important; text-transform:uppercase !important; letter-spacing:1px !important; }
#analyse-btn { background:linear-gradient(135deg,#8A6E30,#C9A84C,#E8C96A,#C9A84C) !important; color:#080C14 !important; font-family:'Outfit',sans-serif !important; font-weight:700 !important; font-size:1rem !important; letter-spacing:1.5px !important; text-transform:uppercase !important; border:none !important; border-radius:8px !important; padding:14px 32px !important; width:100% !important; box-shadow:0 4px 24px rgba(201,168,76,.4) !important; transition:transform .2s,box-shadow .2s !important; cursor:pointer !important; }
#analyse-btn:hover { transform:translateY(-2px) !important; box-shadow:0 8px 32px rgba(201,168,76,.6) !important; }
.tab-nav button { font-family:'Outfit',sans-serif !important; font-size:.82rem !important; font-weight:500 !important; color:var(--text-muted) !important; background:transparent !important; border:none !important; border-bottom:2px solid transparent !important; padding:10px 14px !important; transition:color .2s,border-color .2s !important; }
.tab-nav button.selected,.tab-nav button[aria-selected=true] { color:var(--gold-light) !important; border-bottom-color:var(--gold) !important; }
.out-md,.out-md * { font-family:'Outfit',sans-serif !important; color:var(--text-main) !important; line-height:1.85 !important; }
.out-md h2 { font-family:'Cormorant Garamond',serif !important; color:var(--gold-light) !important; font-size:1.4rem !important; border-bottom:1px solid var(--border); padding-bottom:8px; margin-bottom:14px; }
.out-md h3 { font-family:'Cormorant Garamond',serif !important; color:var(--gold) !important; font-size:1.1rem !important; }
.out-md strong { color:#F0D080 !important; }
.out-md em { color:var(--text-muted) !important; }
.out-md a { color:#5BBCFF !important; }
.out-md table { width:100%; border-collapse:collapse; margin:10px 0; }
.out-md th { background:rgba(201,168,76,.12) !important; color:var(--gold) !important; font-size:.78rem; text-transform:uppercase; letter-spacing:.5px; padding:8px 12px; border:1px solid var(--border); }
.out-md td { color:var(--text-main) !important; padding:7px 12px; border:1px solid rgba(201,168,76,.1); font-size:.87rem; }
.out-md tr:nth-child(even) td { background:rgba(201,168,76,.04) !important; }
.out-md blockquote { border-left:3px solid var(--gold); padding-left:16px; color:#B8A888 !important; font-style:italic; margin:10px 0; font-family:'Cormorant Garamond',serif !important; font-size:1.05rem !important; }
.out-md ul li::marker { color:var(--gold) !important; }
.out-md hr { border-color:var(--border) !important; }
.out-md code { background:rgba(201,168,76,.08) !important; color:var(--gold-light) !important; border-radius:4px; padding:1px 6px; }
.examples-holder td,.gr-samples-table td { color:var(--text-main) !important; background:rgba(201,168,76,.04) !important; font-size:.82rem !important; border-color:var(--border) !important; }
.examples-holder tr:hover td { background:rgba(201,168,76,.1) !important; cursor:pointer; }
.disclaimer { background:rgba(231,76,60,.08); border:1px solid rgba(231,76,60,.25); border-radius:8px; padding:10px 16px; font-size:.77rem; color:#C0A898; text-align:center; margin:10px 0; }
.bfsi-footer { text-align:center; color:var(--text-muted); font-size:.72rem; padding:14px; letter-spacing:1px; }
.bfsi-footer span { color:var(--gold-dim); }
::-webkit-scrollbar{width:5px} ::-webkit-scrollbar-track{background:transparent} ::-webkit-scrollbar-thumb{background:rgba(201,168,76,.25);border-radius:3px}
"""

In [15]:

# ── STEP 6: Sample Profiles ────────────────────────────────────────────
EXAMPLES = [
    ["28", "75000",  "500000",  "15000", "Spouse + 1 Child",  "Buy First Home",             "5-7 Years (Medium-Long)",  "Moderate"],
    ["35", "150000", "2000000", "30000", "Spouse + 2 Kids",   "Children Education + Retire","10-15 Years (Very Long)",  "Moderate"],
    ["45", "250000", "5000000", "0",     "Spouse",            "Early Retirement at 55",     "7-10 Years (Long)",        "Conservative"],
    ["24", "45000",  "100000",  "8000",  "None",              "Build Emergency Fund",       "1-3 Years (Short)",        "Aggressive"],
    ["52", "180000", "8000000", "0",     "2 Adult Children",  "Retirement Corpus",          "7-10 Years (Long)",        "Conservative"],
    ["30", "120000", "800000",  "25000", "Parents + Spouse",  "Start Business",             "< 1 Year (Very Short)",    "Aggressive"],
]

In [16]:
# ── STEP 7: Build Gradio UI ────────────────────────────────────────────
def build_ui() -> gr.Blocks:
    with gr.Blocks(title="BFSI AI Advisor") as demo:
        gr.HTML('<div class="gold-rule"></div>')
        gr.HTML("""
        <div style="text-align:center;padding:20px 0 6px">
          <h1 id="bfsi-title">🏦 BFSI AI ADVISOR</h1>
          <p id="bfsi-subtitle">
            Banking &nbsp;·&nbsp; Finance &nbsp;·&nbsp; Securities &nbsp;·&nbsp; Insurance
            &nbsp;&nbsp;|&nbsp;&nbsp;
            <span>Powered by Groq LLaMA 3.1 — Open-Source LLM</span>
          </p>
        </div>""")
        gr.HTML("""
        <div class="agents-strip">
          <span class="agent-badge">🔍 Profile Analyst</span>
          <span class="agent-badge">💳 Credit Advisor</span>
          <span class="agent-badge">📈 Investment Strategist</span>
          <span class="agent-badge">🛡️ Insurance Consultant</span>
          <span class="agent-badge">🧾 Tax Expert</span>
          <span class="agent-badge">📚 Literacy Coach</span>
        </div>""")

        gr.HTML('<div class="section-label">Personal &amp; Financial Profile</div>')
        with gr.Row():
            age_in         = gr.Textbox(label="🎂 Age (years)",            placeholder="e.g. 32")
            income_in      = gr.Textbox(label="💰 Monthly Income (₹)",     placeholder="e.g. 80000")
            savings_in     = gr.Textbox(label="🏦 Total Savings (₹)",      placeholder="e.g. 500000")
            dependents_in  = gr.Textbox(label="👨‍👩‍👧 Dependents",            placeholder="e.g. Spouse + 2 Kids")

        gr.HTML('<div class="section-label">Goals &amp; Risk Preference</div>')
        with gr.Row():
            debt_in     = gr.Textbox(label="💸 Existing EMI/Debt (₹/month)", placeholder="e.g. 20000  (0 if none)")
            goal_in     = gr.Textbox(label="🎯 Primary Financial Goal",       placeholder="e.g. Buy Home, Retirement…")
            horizon_in  = gr.Dropdown(label="🗓️ Investment Horizon",
                choices=["< 1 Year (Very Short)","1-3 Years (Short)","3-5 Years (Medium)",
                         "5-7 Years (Medium-Long)","7-10 Years (Long)","10-15 Years (Very Long)","15+ Years (Ultra Long)"],
                value="5-7 Years (Medium-Long)")
            risk_in     = gr.Dropdown(label="⚡ Risk Appetite",
                choices=["Conservative","Moderate","Aggressive"], value="Moderate")

        gr.HTML('<div class="disclaimer">⚠️ <strong>Disclaimer:</strong> This AI advisor provides general financial information only. It does not constitute personalised financial advice. Always consult SEBI/IRDAI/RBI registered professionals before making actual financial decisions.</div>')

        with gr.Row():
            with gr.Column(scale=1): gr.HTML("")
            with gr.Column(scale=3):
                analyse_btn = gr.Button("⚡  Analyse My Financial Profile", variant="primary", elem_id="analyse-btn")
            with gr.Column(scale=1): gr.HTML("")

        gr.Examples(
            examples=EXAMPLES,
            inputs=[age_in, income_in, savings_in, debt_in, dependents_in, goal_in, horizon_in, risk_in],
            label="💼 Sample Client Profiles — click any row to auto-fill",
        )

        gr.HTML('<div class="gold-rule" style="margin:20px 0 16px;"></div>')
        gr.HTML('<div class="section-label">Your Comprehensive BFSI Report</div>')

        with gr.Tabs():
            with gr.TabItem("🔍 Profile"):
                profile_out = gr.Markdown(value="*Fill in your profile above and click Analyse …*", elem_classes="out-md")
            with gr.TabItem("💳 Credit & Loans"):
                credit_out  = gr.Markdown(value="*Loan eligibility, credit score tips, and EMI analysis …*", elem_classes="out-md")
            with gr.TabItem("📈 Investments"):
                invest_out  = gr.Markdown(value="*Portfolio allocation, mutual fund picks, SIP recommendations …*", elem_classes="out-md")
            with gr.TabItem("🛡️ Insurance"):
                insure_out  = gr.Markdown(value="*Life, health, term & critical illness recommendations …*", elem_classes="out-md")
            with gr.TabItem("🧾 Tax Planning"):
                tax_out     = gr.Markdown(value="*Tax regime comparison, 80C/80D deductions, savings strategy …*", elem_classes="out-md")
            with gr.TabItem("📚 Action Plan"):
                coach_out   = gr.Markdown(value="*Financial literacy guide, fraud alerts, 30-60-90 day plan …*", elem_classes="out-md")

        gr.HTML('<div class="bfsi-footer">Groq LLaMA 3.1 (Open-Source) &nbsp;·&nbsp; 6-Agent Architecture &nbsp;·&nbsp; Gradio UI &nbsp;·&nbsp; <span>For Educational Purposes Only</span></div>')

        analyse_btn.click(
            fn=bfsi_advisor,
            inputs=[age_in, income_in, savings_in, debt_in, dependents_in, goal_in, horizon_in, risk_in],
            outputs=[profile_out, credit_out, invest_out, insure_out, tax_out, coach_out],
            show_progress="full",
        )
    return demo

In [17]:
# ── STEP 8: Launch (Colab-compatible) ─────────────────────────────────
print("\n" + "═"*60)
print("  🏦  BFSI AI Multi-Agent Advisor")
print(f"  Fast LLM : Groq {GROQ_MODEL}")
print(f"  HQ LLM   : Groq {GROQ_MODEL_HQ}")
print("  Agents   : 6 specialist BFSI agents")
print("  ➜  Launching with public share link ...")
print("═"*60 + "\n")

build_ui().launch(
    share=True,       # ← Required for Colab — generates public gradio.live URL
    show_error=True,
    debug=False
    # css=CSS,          # ← css passed to launch() (Gradio 6 compatible, no deprecation warning) - Removed to fix TypeError
)


════════════════════════════════════════════════════════════
  🏦  BFSI AI Multi-Agent Advisor
  Fast LLM : Groq llama-3.1-8b-instant
  HQ LLM   : Groq llama-3.3-70b-versatile
  Agents   : 6 specialist BFSI agents
  ➜  Launching with public share link ...
════════════════════════════════════════════════════════════

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ff993764921a7db52.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 🔑 Getting Your Free API Keys

**Groq (Required — Open-Source LLM):**
1. Visit https://console.groq.com
2. Sign up free → API Keys → Create Key
3. Paste in Cell 2: `GROQ_API_KEY = "gsk_..."`

**Serper (Optional — enables live market data):**
1. Visit https://serper.dev → Free tier: 2,500 searches/month
2. Paste in Cell 2: `SERPER_API_KEY = "..."`

## ⚠️ Important Disclaimer
This tool is for **educational and demonstration purposes only**.
AI-generated content does not constitute professional financial, investment,
insurance, or tax advice. Always consult SEBI-registered advisors,
IRDAI-licensed agents, and Chartered Accountants for actual decisions.
